# 0. Preamble

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.metrics import explained_variance_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

def plot_feature_target_relationships(df, target_col, feature_cols=None, n_cols=3):
    """
    Create a grid of scatterplots showing relationships between features and target.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing features and target
    target_col : str
        Name of the target column
    feature_cols : list of str, optional
        List of feature column names. If None, uses all columns except target
    n_cols : int, default=3
        Number of columns in the subplot grid

    Returns
    -------
    plotly.graph_objects.Figure
        Plotly figure with subplots
    """
    # Get feature columns if not specified
    if feature_cols is None:
        feature_cols = [col for col in df.columns if col != target_col]

    n_features = len(feature_cols)
    n_rows = int(np.ceil(n_features / n_cols))

    # Create subplots
    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        subplot_titles=feature_cols,
        vertical_spacing=0.12,
        horizontal_spacing=0.08
    )

    # Add scatterplots
    for idx, feature in enumerate(feature_cols):
        row = idx // n_cols + 1
        col = idx % n_cols + 1

        fig.add_trace(
            go.Scatter(
                x=df[feature],
                y=df[target_col],
                mode='markers',
                marker=dict(size=5, opacity=0.6),
                name=feature,
                showlegend=False
            ),
            row=row,
            col=col
        )

        # Update axes labels
        fig.update_xaxes(title_text=feature, row=row, col=col)
        fig.update_yaxes(title_text=target_col, row=row, col=col)

    # Update layout
    fig.update_layout(
        height=300 * n_rows,
        title_text=f"Feature vs {target_col} Relationships",
        showlegend=False
    )

    return fig

# 1. Read the Data

Each sample in this data represents an exercise activity. The intention is to be able to predict the calories burned based on the other characteristics of the exercise and the person exercising.

So let's being by looking at the data to see what we have!

In [ ]:
full = pd.read_csv('calories.csv')
full['Gender'] = full['Gender'].map({'male': 0, 'female': 1})

In [ ]:
plot_feature_target_relationships(full, 'Calories', n_cols=3)

# 2. Here be Dragons

Let's now look at some examples of overfitting starting with building a model with the unique ID's of the individual exercise sessions...

In [ ]:
SAMPLE_SIZE = 50
data = full.sample(SAMPLE_SIZE, random_state=42)
data.head()

In [ ]:
TARGET = 'Calories'
FEATURE = 'User_ID'

X_train, X_test, y_train, y_test = train_test_split(
    data[[c for c in data.columns if c != TARGET]],
    data[TARGET], test_size=0.33, random_state=42
)

In [ ]:
px.scatter(data, x=FEATURE, y=TARGET, height=400, width=400)

There doesn't seem to be much of a relationship, right? Well let's see what happens when we fit a model!

In [ ]:
model = RandomForestRegressor(n_estimators=100)
model.fit(X_train[[FEATURE]], y_train)
print(explained_variance_score(y_train, model.predict(X_train[[FEATURE]])))

So we were able to explain quite a lof of the variance in our data by predicting using the user ID! How is this possible?? Look at what we're evaluating over - the training data! So we are probably overfitting and can check by looking at the testing data.

In [ ]:
print(explained_variance_score(y_test, model.predict(X_test[[FEATURE]])))

Yep! We're definitely overfitting. But it's worth noting that so long as a feature has enough variance it can usually be used to allow the model to "cheat" and pretend like it is super predictive over the training set.

**Try it Yourself!**

Try out different features by replacing the `FEATURES` variable above and rerunning the training and evaluation.

You'll notice that lots of them seem predictive over the training data but are useless over the testing data.

# 3. Adding More Data

With limited data, overfitting is quite hard to avoid and the only real cure is to get more data. So let's do that and see how things change!

In [ ]:
SAMPLE_SIZE = 100
TARGET = 'Calories'
FEATURE = 'Age'


data = full.sample(SAMPLE_SIZE, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    data[[c for c in data.columns if c != TARGET]],
    data[TARGET], test_size=0.33, random_state=42
)

model = RandomForestRegressor(n_estimators=100)
model.fit(X_train[[FEATURE]], y_train)
print(
    'Training Explained Variance:',
    explained_variance_score(y_train, model.predict(X_train[[FEATURE]]))
)
print(
    'Testing Explained Variance:',
    explained_variance_score(y_test, model.predict(X_test[[FEATURE]]))
)


**Try It Yourself**

Keep increasing the data set size (`SAMPLE_SIZE`) and you'll see the difference between the evaluation over the training and testing shrink. Generally more data means it becomes harder and harder to overfit.




# 4. User ID is a Nuisance

However try it where `FEATURE = 'User_ID'`... here the model is able to continually use the unique values no matter how large the data is to pin point the calories per row in the training set.

All to say data with lots of variance makes overfitting a big problem.

In [ ]:
SAMPLE_SIZE = 100
TARGET = 'Calories'
FEATURE = 'User_ID'


data = full.sample(SAMPLE_SIZE, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    data[[c for c in data.columns if c != TARGET]],
    data[TARGET], test_size=0.33, random_state=42
)

model = RandomForestRegressor(n_estimators=100)
model.fit(X_train[[FEATURE]], y_train)
print(
    'Training Explained Variance:',
    explained_variance_score(y_train, model.predict(X_train[[FEATURE]]))
)
print(
    'Testing Explained Variance:',
    explained_variance_score(y_test, model.predict(X_test[[FEATURE]]))
)


# 5. Multiple Features

Overfitting also becomes more likely as you add new features. Here's a feature that works pretty well.

In [ ]:
SAMPLE_SIZE = 250
data = full.sample(SAMPLE_SIZE, random_state=42)

TARGET = 'Calories'
FEATURES = ['Duration']

X_train, X_test, y_train, y_test = train_test_split(
    data[[c for c in data.columns if c != TARGET]],
    data[TARGET], test_size=0.33, random_state=42
)
model = RandomForestRegressor(n_estimators=100)
model.fit(X_train[FEATURES], y_train)
print(
    'Training Explained Variance:',
    explained_variance_score(y_train, model.predict(X_train[FEATURES]))
)
print(
    'Testing Explained Variance:',
    explained_variance_score(y_test, model.predict(X_test[FEATURES]))
)

Yet if we add a second feature we'll see that the score on the test data actually goes down!

In [ ]:
FEATURES = ['Duration', 'Gender']

X_train, X_test, y_train, y_test = train_test_split(
    data[[c for c in data.columns if c != TARGET]],
    data[TARGET], test_size=0.33, random_state=42
)
model = RandomForestRegressor(n_estimators=100)
model.fit(X_train[FEATURES], y_train)
print(
    'Training Explained Variance:',
    explained_variance_score(y_train, model.predict(X_train[FEATURES]))
)
print(
    'Testing Explained Variance:',
    explained_variance_score(y_test, model.predict(X_test[FEATURES]))
)

This is thanks to the fact that we've just added a whole lot more variance to our data, thereby allowing our model to cheat on the training data.

# 6. Hyper Parameter Tuning

We can attempt to control the flexibility of the model to combat this.

In [ ]:
SAMPLE_SIZE = 250
data = full.sample(SAMPLE_SIZE, random_state=42)

TARGET = 'Calories'

X_train, X_test, y_train, y_test = train_test_split(
    data[[c for c in data.columns if c != TARGET]],
    data[TARGET], test_size=0.33, random_state=42
)

FEATURES = ['Duration', 'Gender']

param_grid = {
    'n_estimators': [10, 50, 100],
    'max_depth': [None, 20, 10, 5],
    'min_samples_split': [2, 5, 10],
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid, cv=3, n_jobs=-1,
    verbose=2
)
grid_search.fit(X_train[FEATURES], y_train)

model = grid_search.best_estimator_

In [ ]:
grid_search.best_score_

In [ ]:
print(
    'Training Explained Variance:',
    explained_variance_score(y_train, model.predict(X_train[FEATURES]))
)
print(
    'Testing Explained Variance:',
    explained_variance_score(y_test, model.predict(X_test[FEATURES]))
)

So it's a little better, but still just better to use the one feature. When your data set is small sometimes you just have to throw out features.

This means it's really important you don't just throw all the features you have at a model all at once. Garbage in, garbage out.

# 7. Let's Try This For Real

Go ahead and choose the sample size you'd like. Note if you want a challenge you can make the sample size smaller - really large samples will be really easy to fit.

Then try out the feature you think is most useful, train a model, then look at the residuals to pick your next feature, add it in, and repeat!

In [ ]:
SAMPLE_SIZE = 50
data = full.sample(SAMPLE_SIZE, random_state=42)

TARGET = 'Calories'
FEATURES = ['Age']

X_train, X_test, y_train, y_test = train_test_split(
    data[[c for c in data.columns if c != TARGET]],
    data[TARGET], test_size=0.33, random_state=42
)

param_grid = {
    'n_estimators': [10, 50, 100],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid, cv=3, n_jobs=-1,
    verbose=2
)
grid_search.fit(X_train[FEATURES], y_train)

model = grid_search.best_estimator_

print(
    'Training Explained Variance:',
    explained_variance_score(y_train, model.predict(X_train[FEATURES]))
)
print(
    'Testing Explained Variance:',
    explained_variance_score(y_test, model.predict(X_test[FEATURES]))
)

residuals = data.copy()
residuals['predicted'] = model.predict(data[FEATURES])
residuals['Residual'] = residuals['predicted'] - residuals['Calories']
del residuals['Calories']
del residuals['predicted']
plot_feature_target_relationships(residuals, 'Residual', n_cols=3)

The moral of the story is getting a model to work is about paying close attention to the difference between your training and testing sets, gathering as much data as you can, and then exploring the combinations of features and hyperparameters. No wonder ML only became possible with cheap, accessible compute!

# 8. Another Example

In this case we're going to try and predict average life expectancy for a country given a whole bunch of features about the country.

In [ ]:
# https://www.kaggle.com/datasets/albertovidalrod/gapminder-dataset
data = pd.read_csv('gapminder_data_graphs.csv')
data = data[~np.isnan(data['gdp'])][[c for c in data.columns if c not in ['country', 'continent']]]
print(data.shape)
data.head()

In [ ]:
plot_feature_target_relationships(data, 'life_exp', n_cols=3)

In [ ]:
TARGET = 'life_exp'

# ADD CODE HERE TO SPLIT!

In [ ]:
FEATURES = []

# ADD CODE HERE TO TRAIN!

In [ ]:
# ADD CODE HERE TO GET TRAINING SCORE!

In [ ]:
# ADD CODE HERE TO GET TESTING SCORE!

In [ ]:
residuals = data.copy()
residuals['predicted'] = model.predict(data[FEATURES])
residuals['Residual'] = residuals['predicted'] - residuals[TARGET]
del residuals[TARGET]
del residuals['predicted']
plot_feature_target_relationships(residuals, 'Residual', n_cols=3)

# 9. You Can Use Models to Explore

In [ ]:
FEATURE_OF_INTEREST = 'hdi_index'

_min = data[FEATURE_OF_INTEREST].min()
_max = data[FEATURE_OF_INTEREST].max()
_step = (_max - _min) / 20
df = pd.DataFrame({FEATURE_OF_INTEREST: data[FEATURE_OF_INTEREST].unique()})

for col in data.columns:
    if col not in (FEATURE_OF_INTEREST, TARGET):
        df[col] = data[col].min()

df[TARGET] = model.predict(df[FEATURES])
px.scatter(df, x=FEATURE_OF_INTEREST, y=TARGET, height=400, width=400)